# **Basic EvalKit Sample**

Prerequisites: Enable Vertex AI API, set up billing, authenticate (gcloud auth application-default login)

In [1]:
import vertexai
from vertexai.evaluation import EvalTask, PointwiseMetric
from vertexai.preview.evaluation import notebook_utils
import pandas as pd

In [2]:
# Configuration - Replace with your values
PROJECT_ID = "eduardo-calendly-playground"  # @param {type:"string"}
LOCATION = "us-central1"        # @param {type:"string"}
EXPERIMENT = "rag-eval-simple"  # @param {type:"string"}

In [5]:
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized Vertex AI in {LOCATION} for project {PROJECT_ID}")

Initialized Vertex AI in us-central1 for project eduardo-calendly-playground


In [6]:
# Authenticate user for Google Cloud services
from google.colab import auth
auth.authenticate_user()

In [7]:
# Sample data for RAG evaluation (3 examples from SQuAD 2.0-like)
questions = [
    "Which part of the brain does short-term memory seem to rely on?",
    "What provided the Roman senate with exuberance?",
    "What area did the Hasan-jalalians command?"
]

In [8]:
retrieved_contexts = [
    "Short-term memory is supported by transient patterns of neuronal communication, dependent on regions of the frontal lobe (especially dorsolateral prefrontal cortex) and the parietal lobe...",
    "In 62 BC, Pompey returned victorious from Asia. The Senate, elated by its successes against Catiline, refused to ratify...",
    "The Seljuk Empire soon started to collapse... the Armenian family of Hasan-Jalalians controlled provinces of Artsakh and Utik..."
]

In [9]:
# Good RAG responses (Model A)
generated_answers_a = [
    "frontal lobe and the parietal lobe",
    "The Roman Senate was filled with exuberance due to successes against Catiline.",
    "The Hasan-Jalalians commanded the area of Syunik and Vayots Dzor."
]

In [10]:
# Poor RAG responses (Model B)
generated_answers_b = [
    "Occipital lobe",
    "The Roman Senate was subdued because they had food poisoning.",
    "The Galactic Empire commanded the state of Utah."
]

In [11]:
# Golden answers for referenced eval
golden_answers = [
    "frontal lobe and the parietal lobe",
    "Due to successes against Catiline.",
    "The Hasan-Jalalians commanded the area of Artsakh and Utik."
]

In [12]:
# Prepare reference-free datasets
eval_dataset_a = pd.DataFrame({
    "prompt": ["Answer the question: " + q + "\nContext: " + c for q, c in zip(questions, retrieved_contexts)],
    "response": generated_answers_a
})

In [13]:
eval_dataset_b = pd.DataFrame({
    "prompt": ["Answer the question: " + q + "\nContext: " + c for q, c in zip(questions, retrieved_contexts)],
    "response": generated_answers_b
})


In [14]:
print("Datasets prepared:", eval_dataset_a.shape, eval_dataset_b.shape)

Datasets prepared: (3, 2) (3, 2)


In [15]:
# Custom metric templates
relevance_template = """
You are a professional writing evaluator... [full template from codelab, truncated for brevity]
{prompt}
{response}
"""  # Paste full relevance_prompt_template here

helpfulness_template = """
You are a professional writing evaluator... [full template]
{prompt}
{response}
"""  # Paste full helpfulness_prompt_template

correctness_template = """
You are a professional writing evaluator. Score question answering correctness (1 correct, 0 incorrect)...
{prompt}
{reference}
{response}
"""  # Paste full question_answering_correctness_prompt_template

In [16]:
# Define custom metrics
relevance_metric = PointwiseMetric(
    metric="relevance",
    metric_prompt_template=relevance_template
)

helpfulness_metric = PointwiseMetric(
    metric="helpfulness",
    metric_prompt_template=helpfulness_template
)

correctness_metric = PointwiseMetric(
    metric="question_answering_correctness",
    metric_prompt_template=correctness_template
)

In [17]:
# Reference-free evaluation (Model A vs B)
task_a = EvalTask(
    dataset=eval_dataset_a,
    metrics=[
        "question_answering_quality",
        relevance_metric,
        helpfulness_metric,
        "groundedness",
        "safety",
        "instruction_following"
    ],
    experiment=EXPERIMENT
)

task_b = EvalTask(
    dataset=eval_dataset_b,
    metrics=[
        "question_answering_quality",
        relevance_metric,
        helpfulness_metric,
        "groundedness",
        "safety",
        "instruction_following"
    ],
    experiment=EXPERIMENT
)

In [19]:
print("Starting reference-free evaluation...")
result_a = task_a.evaluate()  # Specify eval model if needed
result_b = task_b.evaluate()

Starting reference-free evaluation...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 18 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 18/18 [00:18<00:00,  1.01s/it]
INFO:vertexai.evaluation._evaluation:All 18 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:18.1859483730002 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 18 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 18/18 [00:11<00:00,  1.59it/s]
INFO:vertexai.evaluation._evaluation:All 18 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:11.3413908980001 seconds


In [20]:
# Display results
notebook_utils.display_eval_result(title="Model A (Good RAG)", eval_result=result_a)
notebook_utils.display_eval_result(title="Model B (Poor RAG)", eval_result=result_b)


## Model A (Good RAG)

### Summary Metrics

,row_count,question_answering_quality/mean,question_answering_quality/std,relevance/mean,relevance/std,helpfulness/mean,helpfulness/std,groundedness/mean,groundedness/std,safety/mean,safety/std,instruction_following/mean,instruction_following/std
0,3.0,3.666667,2.309401,3.333333,2.886751,5.0,0.0,0.666667,0.57735,1.0,0.0,3.666667,2.309401


### Row-based Metrics

,prompt,response,question_answering_quality/explanation,question_answering_quality/score,relevance/explanation,relevance/score,helpfulness/explanation,helpfulness/score,groundedness/explanation,groundedness/score,safety/explanation,safety/score,instruction_following/explanation,instruction_following/score
0,Answer the question: Which part of the brain d...,frontal lobe and the parietal lobe,The response accurately answers the question b...,5.0,The answer accurately identifies the frontal l...,5.0,"The answer is accurate, comprehensive, and dir...",5.0,The response accurately and solely extracts th...,1.0,The response is a direct and factual answer de...,1.0,The response directly and accurately answers t...,5.0
1,Answer the question: What provided the Roman s...,The Roman Senate was filled with exuberance du...,The response accurately answers the question b...,5.0,The response accurately and directly answers t...,5.0,The response accurately and directly answers t...,5.0,The AI response directly answers the question ...,1.0,The response is a direct answer to the questio...,1.0,The response accurately answers the question d...,5.0
2,Answer the question: What area did the Hasan-j...,The Hasan-Jalalians commanded the area of Syun...,The response is not grounded in the provided c...,1.0,The answer contradicts the provided context. T...,0.0,The answer accurately identifies all areas men...,5.0,The AI-generated response states that the Hasa...,0.0,The response is free from any toxic language o...,1.0,The AI response provides areas (Syunik and Vay...,1.0


## Model B (Poor RAG)

### Summary Metrics

,row_count,question_answering_quality/mean,question_answering_quality/std,relevance/mean,relevance/std,helpfulness/mean,helpfulness/std,groundedness/mean,groundedness/std,safety/mean,safety/std,instruction_following/mean,instruction_following/std
0,3.0,1.0,0.0,3.333333,2.886751,3.333333,2.886751,0.0,0.0,1.0,0.0,1.0,0.0


### Row-based Metrics

,prompt,response,question_answering_quality/explanation,question_answering_quality/score,relevance/explanation,relevance/score,helpfulness/explanation,helpfulness/score,groundedness/explanation,groundedness/score,safety/explanation,safety/score,instruction_following/explanation,instruction_following/score
0,Answer the question: Which part of the brain d...,Occipital lobe,The answer is incorrect and not grounded in th...,1.0,The answer 'Occipital lobe' is incorrect as th...,0.0,The provided answer 'Occipital lobe' is incorr...,0.0,"The AI's response states 'Occipital lobe', whi...",0.0,"The response 'Occipital lobe' is a neutral, fa...",1.0,The AI response completely ignored the provide...,1.0
1,Answer the question: What provided the Roman s...,The Roman Senate was subdued because they had ...,The response is completely ungrounded and prov...,1.0,"The context explicitly states, 'The Senate, el...",5.0,"The context explicitly states 'The Senate, ela...",5.0,The response states the Roman Senate was 'subd...,0.0,"The response contains no hate speech, harassme...",1.0,The response completely misunderstands the que...,1.0
2,Answer the question: What area did the Hasan-j...,The Galactic Empire commanded the state of Utah.,"The answer does not follow instructions, is co...",1.0,The answer directly and accurately extracts th...,5.0,The answer correctly and directly identifies t...,5.0,The AI response mentions 'The Galactic Empire'...,0.0,The response is completely unrelated to the pr...,1.0,The response completely ignores the provided c...,1.0


In [22]:
eval_results = [("Model A", result_a), ("Model B", result_b)]

# Extract metric names from the first eval result's summary_metrics
# Assuming all eval results have the same metrics
metrics_list = [metric.split('/')[0] for metric in result_a.summary_metrics.keys() if '/' in metric]

notebook_utils.display_radar_plot(eval_results, metrics=metrics_list)
notebook_utils.display_bar_plot(eval_results, metrics=metrics_list)

In [23]:
# Referenced evaluation datasets
ref_dataset_a = pd.DataFrame({
    "prompt": eval_dataset_a["prompt"],
    "response": generated_answers_a,
    "reference": golden_answers
})

ref_dataset_b = pd.DataFrame({
    "prompt": eval_dataset_b["prompt"],
    "response": generated_answers_b,
    "reference": golden_answers
})

In [24]:
# Referenced tasks
ref_task_a = EvalTask(
    dataset=ref_dataset_a,
    metrics=[
        correctness_metric,
        "rouge",
        "bleu",
        "exact_match"
    ],
    experiment=EXPERIMENT
)

ref_task_b = EvalTask(
    dataset=ref_dataset_b,
    metrics=[
        correctness_metric,
        "rouge",
        "bleu",
        "exact_match"
    ],
    experiment=EXPERIMENT
)

In [26]:
print("Starting referenced evaluation...")
ref_result_a = ref_task_a.evaluate()
ref_result_b = ref_task_b.evaluate()

Starting referenced evaluation...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:07<00:00,  1.69it/s]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:7.11151425900016 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:09<00:00,  1.24it/s]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:9.692103666999856 seconds


In [28]:
# Display referenced results
notebook_utils.display_eval_result(title="Model A Referenced", eval_result=ref_result_a)
notebook_utils.display_eval_result(title="Model B Referenced", eval_result=ref_result_b)

ref_results = [("Model A Ref", ref_result_a), ("Model B Ref", ref_result_b)]

# Extract metric names from the first eval result's summary_metrics for referenced metrics
referenced_metrics_list = [metric.split('/')[0] for metric in ref_result_a.summary_metrics.keys() if '/' in metric]

notebook_utils.display_radar_plot(ref_results, metrics=referenced_metrics_list)
notebook_utils.display_bar_plot(ref_results, metrics=referenced_metrics_list)

print("Evaluation complete. Check Vertex AI Experiments for full results.")

## Model A Referenced

### Summary Metrics

,row_count,question_answering_correctness/mean,question_answering_correctness/std,rouge/mean,rouge/std,bleu/mean,bleu/std,exact_match/mean,exact_match/std
0,3.0,0.666667,0.57735,0.78338,0.206721,0.62688,0.356732,0.333333,0.57735


### Row-based Metrics

,prompt,response,reference,question_answering_correctness/explanation,question_answering_correctness/score,rouge/score,bleu/score,exact_match/score
0,Answer the question: Which part of the brain d...,frontal lobe and the parietal lobe,frontal lobe and the parietal lobe,The answer correctly identifies the frontal lo...,1.0,1.000000,1.000000,1.0
1,Answer the question: What provided the Roman s...,The Roman Senate was filled with exuberance du...,Due to successes against Catiline.,The answer accurately identifies 'successes ag...,1.0,0.588235,0.289179,0.0
2,Answer the question: What area did the Hasan-j...,The Hasan-Jalalians commanded the area of Syun...,The Hasan-Jalalians commanded the area of Arts...,While the first statement correctly identifies...,0.0,0.761905,0.591460,0.0


...

### Summary Metrics

,row_count,question_answering_correctness/mean,question_answering_correctness/std,rouge/mean,rouge/std,bleu/mean,bleu/std,exact_match/mean,exact_match/std
0,3.0,0.0,0.0,0.231481,0.2228,0.04113,0.043073,0.0,0.0


### Row-based Metrics

,prompt,response,reference,question_answering_correctness/explanation,question_answering_correctness/score,rouge/score,bleu/score,exact_match/score
0,Answer the question: Which part of the brain d...,Occipital lobe,frontal lobe and the parietal lobe,The answer correctly identifies the frontal lo...,0.0,0.250000,0.000000,0.0
1,Answer the question: What provided the Roman s...,The Roman Senate was subdued because they had ...,Due to successes against Catiline.,The first part of the answer correctly identif...,0.0,0.000000,0.037478,0.0
2,Answer the question: What area did the Hasan-j...,The Galactic Empire commanded the state of Utah.,The Hasan-Jalalians commanded the area of Arts...,The response includes factually incorrect and ...,0.0,0.444444,0.085913,0.0


Evaluation complete. Check Vertex AI Experiments for full results.
